# Coursework 1
**Replace CID in the file name with your CID**

# Outline


- [Task 1](#task-1): Multi-class classification with Decision Trees and Random Forests <a name="index-task-1"></a>
  - [(1.1)](#task-11) <a name="index-task-11"></a>
  - [(1.2)](#task-12) <a name="index-task-12"></a>
  - [(1.3)](#task-13) <a name="index-task-13"></a>
  - [(1.4)](#task-14) <a name="index-task-14"></a>
- [Task 2](#task-2): Binary classification with the Huberised Support Vector Machine <a name="index-task-2"></a>
  - [(2.1)](#task-21) <a name="index-task-21"></a>
  - [(2.2)](#task-22)  <a name="index-task-22"></a>
- [Task 3](#task-3): Regression with the Multi-Layer Perceptron <a name="index-task-3"></a>
  - [(3.1)](#task-31) <a name="index-task-31"></a>
  - [(3.2)](#task-32)  <a name="index-task-32"></a>



---



<a name="task-1"></a>

# Task 1: Multi-class classification with Decision Trees and Random Forests  [(index)](#index-task-1)

In [4]:
# we first import packages and libraries needed
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# We set a global configuration for matplotlib to obtain well-readable plots
SMALL_SIZE = 12
MEDIUM_SIZE = 16
BIGGER_SIZE = 20

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

# fixing random generator for reproducibility
rng = np.random.default_rng(9001)

<a name="task-11"></a>

## (1.1) [(index)](#index-task-11)

In [5]:
data_train = pd.read_csv('airfield_statistics_test.csv')
data_test = pd.read_csv('airfield_statistics_train.csv')

data_train.head()

,Days of airfrost,Precipitation,Sunshine hours,Humidity (%),Wind Speed,Aircraft total movements,Weather and flight condition category,Runway surface minimal temperature,Runway surface maximal temperature
0,0,29.8,187.7,59.1,2.5,1303,1,12.5,20.9
1,28,12.9,38.4,71.6,5.1,1261,3,-4.6,0.8
2,2,87.0,200.6,56.6,4.0,1345,0,7.2,15.9
3,7,69.7,66.9,57.5,4.4,1241,0,2.8,8.4
4,0,49.6,119.0,59.6,3.7,1341,0,5.8,13.1


In [6]:
data_train.head()

,Days of airfrost,Precipitation,Sunshine hours,Humidity (%),Wind Speed,Aircraft total movements,Weather and flight condition category,Runway surface minimal temperature,Runway surface maximal temperature
0,0,29.8,187.7,59.1,2.5,1303,1,12.5,20.9
1,28,12.9,38.4,71.6,5.1,1261,3,-4.6,0.8
2,2,87.0,200.6,56.6,4.0,1345,0,7.2,15.9
3,7,69.7,66.9,57.5,4.4,1241,0,2.8,8.4
4,0,49.6,119.0,59.6,3.7,1341,0,5.8,13.1


In [10]:
def gini_index(y):
    """
    Arguments:
        y: vector of training labels, of shape (N,).
    Returns:
        (float): the GINI-index for the labels in a given region.
    """

    # Count occurrences of each unique label in y
    unique_labels, counts = np.unique(y, return_counts=True)

    # Compute probabilities for each label
    p = counts / len(y)

    # Compute the sum of squared probabilities
    sum_p_squared = np.sum(p**2)

    # Return GINI-Index
    return 1 - sum_p_squared

# EDIT THIS FUNCTION

def split_samples(X, y, column, value, categorical):
    """
    Return the split of data whose column-th feature:
      1. equals value, in case `column` is categorical, or
      2. less than value, in case `column` is not categorical (i.e. numerical)

    Arguments:
        X: training features, of shape (N, p).
        y: vector of training labels, of shape (N,).
        column: the column of the feature for splitting.
        value: splitting threshold  the samples
        categorical: boolean value indicating whether column is a categorical variable or numerical.
    Returns:
        tuple(np.array, np.array, np.array): tuple of the left split data (X_l, y_l).
        tuple(np.array, np.array, np.array): tuple of the right split data (X_l, y_l)
    """

    if categorical:
        left_mask =(X[:, column] == value)
    else:
        left_mask = (X[:, column] < value)

    # Using the binary masks `left_mask`, we split X, y, and sample_weights.
    X_l, y_l = X[left_mask, :], y[left_mask] # <- SOLUTION
    X_r, y_r = X[~left_mask, :], y[~left_mask] # <- SOLUTION

    return (X_l, y_l), (X_r, y_r)

# EDIT THIS FUNCTION
def gini_split_value(X, y, column, categorical):
    """
    Calculate the GINI-index based on `column` with the split that minimizes the GINI-index.
    Arguments:
        X: training features, of shape (N, p).
        y: vector of training labels, of shape (N,).
        column: the column index of the feature for calculating. 0 <= column < p
        categorical: boolean value indicating whether column is a categorical variable or numerical.
    Returns:
        (float, float): the resulted GINI-index and the corresponding value used in splitting.
    """

    unique_vals = np.unique(X[:, column])

    assert len(unique_vals) > 1, f"There must be more than one distinct feature value. Column: {column}."

    gini_index_val, threshold = np.inf, None

    # iterate over all unique values of the i-th feature
    for value in unique_vals:
        # Split the samples into left and right subsets based on the current value
        (X_l, y_l), (X_r, y_r) = split_samples(X, y, column, value, categorical) ## <-- SOLUTION

        # check if one of the subsets is empty (invalid split)
        if len(y_l) == 0 or len(y_r) == 0:
            continue

        # calculate the proportion of samples on the left and right sides
        p_left = len(y_l)/(len(y_l)+len(y_r))
        p_right = len(y_r)/(len(y_l)+len(y_r))

        # compute the GINI index for the left and right subsets
        new_cost = p_left * gini_index(y_l) + p_right * gini_index(y_r) ## <-- SOLUTION

        # update the minimum GINI index and the threshold if this split is better
        if new_cost < gini_index_val:
              gini_index_val, threshold = new_cost, value

    return gini_index_val, threshold

## EDIT THIS FUNCTION
def gini_split(X, y, cat_columns_dict):
    """
    Choose the best feature to split according to criterion.
    Args:
        X: training features, of shape (N, p).
        y: vector of training labels, of shape (N,).
        cat_columns_dict: a dictionary mapping column indices to whether the column is categorical or numerical variable.
    Returns:
        (float, int, float): the minimal Gini index found, the best feature index and value used in splitting.
        If the feature index is None, then no valid split for the current Node.
    """

    # Initialize `split_column` to None, so if None returned this means there is no valid split at the current node.
    min_gini_index = np.inf
    split_column = None
    split_val = np.nan

    for column, categorical in cat_columns_dict.items():
        # skip column if samples are not separable by that column.
        if len(np.unique(X[:, column])) < 2:
            continue
        gini_index, current_split_val = gini_split_value(X, y, column, categorical)  ## <-- SOLUTION


        if gini_index < min_gini_index: ## <-- SOLUTION
            # Keep track with:

            # 1. the current minimum gini-index value,
            min_gini_index = gini_index ## <-- SOLUTION

            # 2. corresponding column,
            split_column = column ## <-- SOLUTION

            # 3. corresponding split threshold.
            split_val = current_split_val ## <-- SOLUTION

    return min_gini_index, split_column, split_val

def majority_vote(y):
    """
    Return the label which appears the most in y.
    Args:
        y: vector of training labels, of shape (N,).
    Returns:
        (int): the majority label
    """
    # Find unique labels and their counts
    labels, counts = np.unique(y, return_counts=True)

    # Identify the label with the maximum count
    majority_label = labels[np.argmax(counts)]

    return majority_label

# EDIT THIS FUNCTION
def build_tree(X, y, cat_columns_dict, feature_names, depth, max_depth=10, min_samples_leaf=2):
    """Build the decision tree according to the data.
    Args:
        X: (np.array) training features, of shape (N, p).
        y: (np.array) vector of training labels, of shape (N,).
        cat_columns_dict: a dictionary mapping column indices to whether the column is categorical or numerical variable.
        feature_names (list): record the name of features in X in the original dataset.
        depth (int): current depth for this node.
        max_depth (int): maximum depth of the tree.
        min_samples_leaf (int): minimum number of samples in a leaf node.
    Returns:
        (dict): a dict denoting the decision tree (binary-tree). Each parent node has seven attributes:
          1. 'feature_name': The column name of the split.
          2. 'feature_index': The column index of the split.
          3. 'value': The value used for the split.
          4. 'categorical': indicator for categorical/numerical variables.
          5. 'majority_label': For leaf nodes, this stores the dominant label. Otherwise, it is None.
          6. 'left': The left sub-tree with the same structure.
          7. 'right' The right sub-tree with the same structure.
    """
    # include a clause for the cases where (i) no feature, (ii) all labels are the same,
    # (iii) depth exceeded, or (iv) X is too small
    if len(np.unique(y))==1 or depth>max_depth or len(X)<=min_samples_leaf:
        return {'majority_label': majority_vote(y)}

    GI, split_index, split_val = gini_split(X, y, cat_columns_dict)  ## <-- SOLUTION

    # If no valid split at this node, use majority vote.
    if split_index is None:
        return {'majority_label': majority_vote(y)}

    categorical = cat_columns_dict[split_index]

    # Split samples (X, y, sample_weights) given column, split-value, and categorical flag.
    (X_l, y_l), (X_r, y_r) = split_samples(X, y, split_index, split_val, categorical) ## <-- SOLUTION
    return {
        'feature_name': feature_names[split_index],
        'feature_index': split_index,
        'value': split_val,
        'categorical': categorical,
        'majority_label': None,
        'left': build_tree(X_l, y_l, cat_columns_dict, feature_names, depth + 1, max_depth, min_samples_leaf),
        'right': build_tree(X_r, y_r, cat_columns_dict, feature_names, depth + 1, max_depth, min_samples_leaf)
    }

In [ ]:
import numpy as np

def gini_index(y):
    """
    Arguments:
        y: vector of training labels, of shape (N,).
    Returns:
        (float): the GINI-index for the labels in a given region.
    """
    unique_labels, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1 - np.sum(p**2)

def split_samples(X, y, column, value, categorical):
    """
    Return the split of data whose column-th feature:
      1. equals value, in case `column` is categorical, or
      2. less than value, in case `column` is not categorical (i.e. numerical)
    """
    if categorical:
        left_mask = (X[:, column] == value)
    else:
        left_mask = (X[:, column] < value)
    return (X[left_mask], y[left_mask]), (X[~left_mask], y[~left_mask])

def gini_split_value(X, y, column, categorical):
    """
    Calculate the GINI-index based on `column` with the split that minimizes the GINI-index.
    """
    unique_vals = np.unique(X[:, column])
    if len(unique_vals) <= 1:
        return np.inf, None
    gini_index_val, threshold = np.inf, None
    for value in unique_vals:
        (X_l, y_l), (X_r, y_r) = split_samples(X, y, column, value, categorical)
        if len(y_l) == 0 or len(y_r) == 0:
            continue
        p_left = len(y_l) / (len(y_l) + len(y_r))
        p_right = len(y_r) / (len(y_l) + len(y_r))
        new_cost = p_left * gini_index(y_l) + p_right * gini_index(y_r)
        if new_cost < gini_index_val:
            gini_index_val, threshold = new_cost, value
    return gini_index_val, threshold

def gini_split(X, y, cat_columns_dict):
    """
    Choose the best feature to split according to the criterion.
    """
    min_gini_index = np.inf
    split_column, split_val = None, np.nan
    for column, categorical in cat_columns_dict.items():
        if len(np.unique(X[:, column])) < 2:
            continue
        gini_index, current_split_val = gini_split_value(X, y, column, categorical)
        if gini_index < min_gini_index:
            min_gini_index, split_column, split_val = gini_index, column, current_split_val
    return min_gini_index, split_column, split_val

def majority_vote(y):
    """
    Return the label which appears the most in y.
    """
    labels, counts = np.unique(y, return_counts=True)
    return labels[np.argmax(counts)]

def build_tree(X, y, cat_columns_dict, feature_names, depth, max_depth=10, min_samples_leaf=12):
    """
    Build the decision tree according to the data.
    """
    if len(np.unique(y)) == 1 or depth > max_depth or len(X) <= min_samples_leaf:
        return {'majority_label': majority_vote(y)}
    GI, split_index, split_val = gini_split(X, y, cat_columns_dict)
    if split_index is None:
        return {'majority_label': majority_vote(y)}
    categorical = cat_columns_dict[split_index]
    (X_l, y_l), (X_r, y_r) = split_samples(X, y, split_index, split_val, categorical)
    return {
        'feature_name': feature_names[split_index],
        'feature_index': split_index,
        'value': split_val,
        'categorical': categorical,
        'majority_label': None,
        'left': build_tree(X_l, y_l, cat_columns_dict, feature_names, depth + 1, max_depth, min_samples_leaf),
        'right': build_tree(X_r, y_r, cat_columns_dict, feature_names, depth + 1, max_depth, min_samples_leaf)
    }

# Load dataset (replace with actual dataset loading)
X = np.random.rand(500, 5)  # 500 samples, 5 features
y = np.random.randint(0, 3, 500)  # 3-class classification
feature_names = [f'Feature {i}' for i in range(X.shape[1])]
cat_columns_dict = {i: False for i in range(X.shape[1])}  # Assuming all features are numerical

decision_tree = build_tree(X, y, cat_columns_dict, feature_names, depth=0, max_depth=10, min_samples_leaf=12)
print(decision_tree)


<a name="task-12"></a>

## (1.2) [(index)](#index-task-12)

<a name="task-13"></a>

## (1.3) [(index)](#index-task-13)

<a name="task-14"></a>

## (1.4) [(index)](#index-task-14)

<a name="task-2"></a>

# Task 2: Binary classification with the Huberised Support Vector Machine [(index)](#index-task-2)

<a name="task-21"></a>

## (2.1) [(index)](#index-task-21)

<a name="task-22"></a>

## (2.2) [(index)](#index-task-22)

<a name="task-3"></a>

# Task 3: Regression with the Multi-Layer Perceptron [(index)](#index-task-3)

<a name="task-31"></a>

## (3.1) [(index)](#index-task-31)

<a name="task-32"></a>

## (3.2) [(index)](#index-task-32)